# Hybrid Movie Recommendation System



## Import the libraries

First, we import the tools used in this project.

In [1]:
from __future__ import annotations

import json
import re
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore', category=UserWarning)


## Controlling Recommendations:

These values control the system. The quality weight gives a small advantage to well-rated and popular movies, while still keeping content similarity as the main decision.

In [2]:
N_CANDIDATES = 200
QUALITY_WEIGHT = 0.15
DEFAULT_MMR_LAMBDA = 0.80


In [3]:
# Bounds-checking so a bad value fails loudly instead of quietly breaking recommendations
assert 0 <= QUALITY_WEIGHT <= 1, "QUALITY_WEIGHT must be between 0 and 1"
assert 0 <= DEFAULT_MMR_LAMBDA <= 1, "DEFAULT_MMR_LAMBDA must be between 0 and 1"


 Reasons of such values of **QUALITY_WEIGHT** and **DEFAULT_MMR_LAMBDA**:

=> Cosine similarity ranges 0-1 therefore we keep quality's influence secondary.                             
=> Prioritize relevance is at 20% as diversity avoids duplicate-feel results.

## Data Loading:



In [4]:
movies_raw = pd.read_csv('tmdb_5000_movies.csv')
credits_raw = pd.read_csv('tmdb_5000_credits.csv')

print('Movies table shape:', movies_raw.shape)
print('Credits table shape:', credits_raw.shape)


Movies table shape: (4803, 20)
Credits table shape: (4803, 4)


## Viewing our Data:

The second dataset has the following features:-

*budget* - The budget in which the movie was made.  
*genre* - The genre of the movie, Action, Comedy ,Thriller etc.  
*homepage* - A link to the homepage of the movie.  
*id* - This is infact the movie_id as in the first dataset.  
*keywords* - The keywords or tags related to the movie.  
*original_language* - The language in which the movie was made.  
*original_title* - The title of the movie before translation or adaptation.  
*overview* - A brief description of the movie.  
*popularity* - A numeric quantity specifying the movie popularity.  
*production_companies* - The production house of the movie.  
*production_countries* - The country in which it was produced.  
*release_date*- The date on which it was released.  
*revenue*- The worldwide revenue generated by the movie.  
*runtime* - The running time of the movie in minutes.  
*status* - "Released" or "Rumored".  
*tagline* - Movie's tagline.  
*title* - Title of the movie.  
*vote_average* - average ratings the movie recieved.  
*vote_count* - the count of votes recieved.

The second dataset contains the following features:-

*movie_id* - A unique identifier for each movie.  
*title* - Title of the movie.  
*cast* - The name of lead and supporting actors.  
*crew* - The name of Director, Editor, Composer, Writer etc. 

In [5]:
display(movies_raw.head(3))
print('Movies columns:', movies_raw.columns.tolist())

display(credits_raw.head(3))
print('Credits columns:', credits_raw.columns.tolist())


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466


Movies columns: ['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']


,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."


Credits columns: ['movie_id', 'title', 'cast', 'crew']


### Merging both the datasets:  
We merge using TMDB's numerical movie ID.

In [6]:
credits_raw = credits_raw.rename(columns={'movie_id': 'id', 'title': 'credit_title'})
movies = movies_raw.merge(
    credits_raw[['id', 'cast', 'crew']],
    on='id', how='left',validate='one_to_one'
)

# One row per movie makes later matrix rows and dataframe rows match exactly.
movies = movies.drop_duplicates('id').reset_index(drop=True).copy()
print(f'{len(movies):,} unique movies loaded')
movies[['id', 'title', 'release_date', 'vote_average', 'vote_count']].head()


4,803 unique movies loaded


,id,title,release_date,vote_average,vote_count
0,19995,Avatar,2009-12-10,7.2,11800
1,285,Pirates of the Caribbean: At World's End,2007-05-19,6.9,4500
2,206647,Spectre,2015-10-26,6.3,4466
3,49026,The Dark Knight Rises,2012-07-16,7.6,9106
4,49529,John Carter,2012-03-07,6.1,2124


In [7]:
# Report merge quality instead of assuming it's clean
missing_credits = movies['cast'].isna().sum()
print(f'{missing_credits} movies have no credits data after merge')


0 movies have no credits data after merge


In [8]:
# Confirm the merge didn't silently drop or duplicate rows
assert len(movies) == movies['id'].nunique(), "Duplicate movie IDs found after merge"
assert movies['cast'].notna().mean() > 0.95, "Too many movies missing cast/crew data"


## Feature Extraction:



In [9]:
_parse_failures = 0

def parse_json_list(value):
    global _parse_failures
    if not isinstance(value, str) or not value.strip():
        return []
    try:
        parsed = json.loads(value)
    except json.JSONDecodeError:
        _parse_failures += 1
        return []
    return parsed if isinstance(parsed, list) else []

print(f'{_parse_failures} cells failed JSON parsing')


0 cells failed JSON parsing


In [10]:
def names(value, limit=None):

    result = [item['name'].strip() for item in parse_json_list(value)
              if isinstance(item, dict) and item.get('name', '').strip()]
    return result[:limit] if limit else result
    

In [11]:
def crew_names(value, jobs):

    return [item['name'] for item in parse_json_list(value)
            if isinstance(item, dict) and item.get('job') in jobs and item.get('name')]
    

Peeking at the dataset before and after cleaning.

In [12]:
movies.loc[0, ['title', 'genres', 'cast']]


title                                                Avatar
genres    [{"id": 28, "name": "Action"}, {"id": 12, "nam...
cast      [{"cast_id": 242, "character": "Jake Sully", "...
Name: 0, dtype: object

### Applying the earlier helper functions across the dataset:


In [13]:
movies['year'] = pd.to_datetime(movies['release_date'], errors='coerce').dt.year
missing_year = movies['year'].isna().sum()
print(f'{missing_year} movies have no valid release year')


1 movies have no valid release year


In [14]:
movies['overview'] = movies['overview'].fillna('')
movies['tagline'] = movies['tagline'].fillna('')


In [15]:
def extract_crew(value):
    parsed = parse_json_list(value)
    return {
        'director_list': [i['name'] for i in parsed if isinstance(i, dict) and i.get('job') == 'Director' and i.get('name')],
        'writer_list': [i['name'] for i in parsed if isinstance(i, dict) and i.get('job') in {'Writer','Screenplay','Story'} and i.get('name')],
    }

crew_extracted = movies['crew'].map(extract_crew)
movies['director_list'] = crew_extracted.map(lambda d: d['director_list'])
movies['writer_list'] = crew_extracted.map(lambda d: d['writer_list'])


In [16]:
movies['genres_list'] = movies['genres'].map(names)
movies['keywords_list'] = movies['keywords'].map(names)
movies['cast_list'] = movies['cast'].map(lambda value: names(value, limit=5))


movies[['title', 'genres_list', 'cast_list', 'director_list', 'year']].head()


,title,genres_list,cast_list,director_list,year
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[Sam Worthington, Zoe Saldana, Sigourney Weave...",[James Cameron],2009.0
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[Johnny Depp, Orlando Bloom, Keira Knightley, ...",[Gore Verbinski],2007.0
2,Spectre,"[Action, Adventure, Crime]","[Daniel Craig, Christoph Waltz, Léa Seydoux, R...",[Sam Mendes],2015.0
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[Christian Bale, Michael Caine, Gary Oldman, A...",[Christopher Nolan],2012.0
4,John Carter,"[Action, Adventure, Science Fiction]","[Taylor Kitsch, Lynn Collins, Samantha Morton,...",[Andrew Stanton],2012.0


## Tokenisation:

Structured values become tokens such as `genre_science_fiction` and `director_christopher_nolan`. Keeping a prefix tells the model what type of information each word represents.

In [17]:
def tokenise(values, prefix):
    
    cleaned = [re.sub(r'[^a-z0-9]+', '_', str(value).lower()).strip('_') for value in values]
    return ' '.join(f'{prefix}_{value}' for value in cleaned if value)
    

In [18]:
movies['text_document'] = (movies['overview'] + ' ' + movies['tagline']).str.strip()
movies['genre_document'] = movies['genres_list'].map(lambda values: tokenise(values, 'genre'))
movies['keyword_document'] = movies['keywords_list'].map(lambda values: tokenise(values, 'keyword'))
movies['people_document'] = (
    movies['cast_list'].map(lambda values: tokenise(values, 'cast')) + ' ' +
    movies['director_list'].map(lambda values: tokenise(values, 'director')) + ' ' +
    movies['writer_list'].map(lambda values: tokenise(values, 'writer'))
).str.strip()


In [19]:
movies[['title', 'text_document', 'genre_document', 'people_document']].head(2)


,title,text_document,genre_document,people_document
0,Avatar,"In the 22nd century, a paraplegic Marine is di...",genre_action genre_adventure genre_fantasy gen...,cast_sam_worthington cast_zoe_saldana cast_sig...
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",genre_adventure genre_fantasy genre_action,cast_johnny_depp cast_orlando_bloom cast_keira...


## Vectorization:

TF-IDF converts words into meaningful numerical features. Common words receive less importance; words and phrases that help distinguish a movie receive more importance. I have used separate feature blocks to preserve the importance of plot, genres, keywords, and people.

In [20]:
text_vectorizer = TfidfVectorizer(
    stop_words='english', ngram_range=(1, 2), min_df=2, max_df=0.90,
    sublinear_tf=True, max_features=19_000
)

## print('Actual vocabulary size:', len(text_vectorizer.vocabulary_))
## Actual vocabulary size: 19059


In [21]:
meta_vectorizer = TfidfVectorizer(
    token_pattern=r'(?u)\b[\w_]+\b', min_df=1, sublinear_tf=True
)


In [22]:
meta_vectorizer_genre = TfidfVectorizer(token_pattern=r'(?u)\b[\w_]+\b', min_df=1, sublinear_tf=True)
meta_vectorizer_keyword = TfidfVectorizer(token_pattern=r'(?u)\b[\w_]+\b', min_df=1, sublinear_tf=True)
meta_vectorizer_people = TfidfVectorizer(token_pattern=r'(?u)\b[\w_]+\b', min_df=1, sublinear_tf=True)

X_text = text_vectorizer.fit_transform(movies['text_document'])

X_genre = meta_vectorizer_genre.fit_transform(movies['genre_document'])
X_keyword = meta_vectorizer_keyword.fit_transform(movies['keyword_document'])
X_people = meta_vectorizer_people.fit_transform(movies['people_document'])


In [23]:
print('Plot features:', X_text.shape)
print('Genre features:', X_genre.shape)
print('Keyword features:', X_keyword.shape)
print('People features:', X_people.shape)


Plot features: (4803, 19000)
Genre features: (4803, 20)
Keyword features: (4803, 9804)
People features: (4803, 16546)


## Final Hybrid Feature Matrix

Genres and keywords are given a little more importance than the cast and crew.


In [24]:
X = hstack([
    X_text * 1.00,
    X_genre * 1.40,
    X_keyword * 1.15,
    X_people * 0.75,
], format='csr')
X


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 236432 stored elements and shape (4803, 45370)>

### L2 Normalization:

In [25]:
row_norm = np.sqrt(X.multiply(X).sum(axis=1)).A1
X = X.multiply(1 / np.maximum(row_norm, 1e-12)[:, None]).tocsr()


In [26]:
row_norm = np.sqrt(X.multiply(X).sum(axis=1)).A1
zero_rows = (row_norm < 1e-10).sum()
if zero_rows > 0:
    print(f'Warning: {zero_rows} movies have an all-zero feature vector (no text/genre/cast/keyword data)')
X = X.multiply(1 / np.maximum(row_norm, 1e-12)[:, None]).tocsr()


In [27]:
# These dictionaries help us find a movie quickly from its title or TMDB ID.
title_to_indices = movies.groupby(movies['title'].str.casefold()).indices
id_to_index = pd.Series(movies.index, index=movies['id']).to_dict()

print(f'Final index shape: {X.shape}')
print(f'Index density: {X.nnz / (X.shape[0] * X.shape[1]):.6%}')


Final index shape: (4803, 45370)
Index density: 0.108499%


## Bayesian Rating Balance:

Reliable movies are given a small quality advantage. A movie with one 10/10 rating should not beat a movie with thousands of ratings. Bayesian rating balances each movie's rating with the average rating across the dataset. We combine that rating with popularity to create a small quality score.

In [28]:
def bayesian_rating(frame, quantile=0.60):
    """Calculate a rating that accounts for both score and number of votes."""
    votes = frame['vote_count'].fillna(0).astype(float)
    rating = frame['vote_average'].fillna(0).astype(float)
    minimum_votes = votes.quantile(quantile)
    votes_with_ratings = votes[votes > 0]
    global_mean = rating.loc[votes > 0].mean() if len(votes_with_ratings)>0 else 5.0
    return (votes / (votes + minimum_votes)) * rating + (minimum_votes / (votes + minimum_votes)) * global_mean

movies['bayesian_rating'] = bayesian_rating(movies)


In [29]:
# Scale both values to the same 0-to-1 range before combining them.
popularity = np.log1p(movies['popularity'].fillna(0).clip(lower=0)).to_numpy().reshape(-1, 1)
rating = movies['bayesian_rating'].to_numpy().reshape(-1, 1)
quality = (
    0.75 * MinMaxScaler().fit_transform(rating).ravel() +
    0.25 * MinMaxScaler().fit_transform(popularity).ravel()
)
movies[['title', 'vote_average', 'vote_count', 'bayesian_rating']].head()


,title,vote_average,vote_count,bayesian_rating
0,Avatar,7.2,11800,7.168725
1,Pirates of the Caribbean: At World's End,6.9,4500,6.844650
2,Spectre,6.3,4466,6.290190
3,The Dark Knight Rises,7.6,9106,7.544207
4,John Carter,6.1,2124,6.110663


In [30]:
# Sanity-check the quality score before trusting it in final recommendations
assert quality.min() >= 0 and quality.max() <= 1, "quality score out of expected [0,1] range"
assert not np.isnan(quality).any(), "quality score contains NaN values"


## Pipeline stage:

These small functions solve repeated tasks: locating a movie from its title, selecting varied results, and explaining why a particular movie was recommended.

In [31]:
def resolve_title(title, year=None):
    if not isinstance(title, str) or not title.strip():
        raise ValueError('Title must be a non-empty string')

    matches = list(title_to_indices.get(title.casefold(), []))
    if year is not None:
        matches = [index for index in matches if movies.at[index, 'year'] == year]

    if not matches:
        sample = movies.loc[
            movies['title'].str.contains(title, case=False, na=False, regex=False),
            ['title', 'year']
        ].head(8)
        if sample.empty:
            raise ValueError(f'No title found matching: {title!r} — check spelling.')
        raise ValueError(f'No exact title found: {title!r}. Possible matches:\n{sample.to_string(index=False)}')

    # A title may have remakes; default deterministically to the most-voted one.
    return max(matches, key=lambda index: movies.at[index, 'vote_count'])
    

In [32]:
def _explanation(source_idx, candidate_idx):
    
    shared_genres = sorted(set(movies.at[source_idx, 'genres_list']) & set(movies.at[candidate_idx, 'genres_list']))
    shared_cast = sorted(set(movies.at[source_idx, 'cast_list']) & set(movies.at[candidate_idx, 'cast_list']))
    source_director = set(movies.at[source_idx, 'director_list'])
    candidate_director = set(movies.at[candidate_idx, 'director_list'])
    
    reasons = []
    if shared_genres: reasons.append('shared genres: ' + ', '.join(shared_genres[:2]))
    if shared_cast: reasons.append('shared cast: ' + ', '.join(shared_cast[:2]))
    if source_director & candidate_director: reasons.append('same director: ' + ', '.join(source_director & candidate_director))

    return '; '.join(reasons) if reasons else 'similar plot/theme vocabulary'


## Recommendations with diversity:

MMR (Maximal Marginal Relevance) stops the list from filling with almost identical movies. It selects a relevant movie, then slightly penalises candidates that are too similar to results already selected.

In [33]:
def mmr_select(candidates, scores, n=10, diversity=0.20):
    
    selected = []
    remaining = list(candidates)
    score_by_idx = dict(zip(candidates, scores))

    while remaining and len(selected) < n:
        if not selected:
            chosen = max(remaining, key=score_by_idx.get)
        else:
            selected_matrix = X[selected]
            chosen = max(
                remaining,
                key=lambda index: (1 - diversity) * score_by_idx[index]
                - diversity * cosine_similarity(X[index], selected_matrix).max()
            )
        selected.append(chosen)
        remaining.remove(chosen)
    return selected


## The Main `recommend()` function:

This function finds a selected movie, calculates content similarity with every other movie, adds the small quality score, considers the best candidates, and returns diverse final recommendations.

In [34]:
def recommend(title, n=10, year=None, diversity=0.20, min_year=None, max_year=None):
    source_idx = resolve_title(title, year)

    # Compare just this movie with all movies. We do not save an all-pairs matrix.
    content_scores = (X[source_idx] @ X.T).toarray().ravel()
    final_scores = (1 - QUALITY_WEIGHT) * content_scores + QUALITY_WEIGHT * quality

    # Re-rank only strong candidates for efficient, varied recommendations.
    candidate_idx = np.argpartition(final_scores, -N_CANDIDATES)[-N_CANDIDATES:]
    candidate_idx = [index for index in candidate_idx if index != source_idx]

    # Optional filters let us limit recommendations by release year.
    if min_year is not None:
        candidate_idx = [index for index in candidate_idx if movies.at[index, 'year'] >= min_year]
    if max_year is not None:
        candidate_idx = [index for index in candidate_idx if movies.at[index, 'year'] <= max_year]

    chosen = mmr_select(candidate_idx, final_scores[candidate_idx], n=n, diversity=diversity)

    result = movies.loc[chosen, ['id', 'title', 'year', 'vote_average', 'vote_count', 'genres_list']].copy()
    result['content_similarity'] = content_scores[chosen]
    result['final_score'] = final_scores[chosen]
    result['why_recommended'] = [_explanation(source_idx, index) for index in chosen]
    return result.sort_values('final_score', ascending=False).reset_index(drop=True)


## Recommendation Test:


In [35]:
recommend('The Avengers', n=10, diversity=0.25)


,id,title,year,vote_average,vote_count,genres_list,content_similarity,final_score,why_recommended
0,99861,Avengers: Age of Ultron,2015.0,7.3,6767,"[Action, Adventure, Science Fiction]",0.711099,0.711878,"shared genres: Action, Adventure; shared cast:..."
1,100402,Captain America: The Winter Soldier,2014.0,7.6,5764,"[Action, Adventure, Science Fiction]",0.589302,0.612529,"shared genres: Action, Adventure; shared cast:..."
2,102899,Ant-Man,2015.0,7.0,5880,"[Science Fiction, Action, Adventure]",0.588753,0.599189,"shared genres: Action, Adventure"
3,271110,Captain America: Civil War,2016.0,7.1,7241,"[Adventure, Action, Science Fiction]",0.571794,0.590423,"shared genres: Action, Adventure; shared cast:..."
4,10138,Iron Man 2,2010.0,6.6,6849,"[Adventure, Action, Science Fiction]",0.575722,0.575325,"shared genres: Action, Adventure; shared cast:..."
5,118340,Guardians of the Galaxy,2014.0,7.9,9742,"[Action, Science Fiction, Adventure]",0.496154,0.552767,"shared genres: Action, Adventure"
6,1771,Captain America: The First Avenger,2011.0,6.6,7047,"[Action, Adventure, Science Fiction]",0.535819,0.541223,"shared genres: Action, Adventure; shared cast:..."
7,36657,X-Men,2000.0,6.8,4097,"[Adventure, Action, Science Fiction]",0.510896,0.510420,"shared genres: Action, Adventure"
8,1724,The Incredible Hulk,2008.0,6.1,3021,"[Science Fiction, Action, Adventure]",0.505395,0.501342,"shared genres: Action, Adventure"
9,11,Star Wars,1977.0,8.1,6624,"[Adventure, Action, Science Fiction]",0.411299,0.477759,"shared genres: Action, Adventure"


## Recommendations from Liked Movies:

This version combines the feature vectors of several films the user likes, then recommends films that fit the combined profile. This is a simple form of personalisation that works even without user-rating data.

In [36]:
def recommend_from_likes(titles, n=10, diversity=0.25):
    if not titles:
        raise ValueError('Provide at least one liked title.')

    liked = [resolve_title(title) for title in titles]
    profile = X[liked].mean(axis=0)
    content_scores = (profile @ X.T).A1
    final_scores = (1 - QUALITY_WEIGHT) * content_scores + QUALITY_WEIGHT * quality

    candidate_idx = np.argpartition(final_scores, -N_CANDIDATES)[-N_CANDIDATES:]
    candidate_idx = [index for index in candidate_idx if index not in liked]
    chosen = mmr_select(candidate_idx, final_scores[candidate_idx], n=n, diversity=diversity)

    result = movies.loc[chosen, ['id', 'title', 'year', 'vote_average', 'genres_list']].copy()
    result['score'] = final_scores[chosen]
    return result.sort_values('score', ascending=False).reset_index(drop=True)

recommend_from_likes(['The Dark Knight', 'Inception', 'Interstellar'], n=10)


,id,title,year,vote_average,genres_list,score
0,49026,The Dark Knight Rises,2012.0,7.6,"[Action, Crime, Drama, Thriller]",0.396362
1,76341,Mad Max: Fury Road,2015.0,7.2,"[Action, Adventure, Science Fiction, Thriller]",0.351176
2,118340,Guardians of the Galaxy,2014.0,7.9,"[Action, Science Fiction, Adventure]",0.349624
3,286217,The Martian,2015.0,7.6,"[Drama, Adventure, Science Fiction]",0.333697
4,119450,Dawn of the Planet of the Apes,2014.0,7.3,"[Science Fiction, Action, Drama, Thriller]",0.330746
5,78,Blade Runner,1982.0,7.9,"[Science Fiction, Drama, Thriller]",0.312349
6,62,2001: A Space Odyssey,1968.0,7.9,"[Science Fiction, Mystery, Adventure]",0.311347
7,98,Gladiator,2000.0,7.9,"[Action, Drama, Adventure]",0.302178
8,1124,The Prestige,2006.0,8.0,"[Drama, Mystery, Thriller]",0.299514
9,58574,Sherlock Holmes: A Game of Shadows,2011.0,7.0,"[Adventure, Action, Crime, Mystery]",0.285594


## Persistence (Saving the Trained Model):



In [37]:
artifact = {
    'movies': movies,
    'matrix': X,
    'text_vectorizer': text_vectorizer,
    'meta_vectorizer': meta_vectorizer,
    'quality': quality,
    'config': {
        'quality_weight': QUALITY_WEIGHT,
        'default_mmr_lambda': DEFAULT_MMR_LAMBDA,
        'n_candidates': N_CANDIDATES,
    },

}

joblib.dump(artifact, 'hybrid_movie_recommender.joblib', compress=3)
print('Saved hybrid_movie_recommender.joblib')


Saved hybrid_movie_recommender.joblib
